# 02 — Production Semantic & Token-Aware Chunking

Docling JSON → semantic/structure-aware chunks → token-aware splitting → multimodal evidence → validation → chunk store.

In [1]:
from pathlib import Path
import json, re, hashlib
from typing import Any, Dict, List
from transformers import AutoTokenizer

print("Notebook 02 ready")

c:\Users\MD RAJA\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Notebook 02 ready


In [2]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed_papers"
CHUNKS_DIR = PROJECT_ROOT / "data" / "chunks"
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

print("Processed:", PROCESSED_DIR)
print("Chunks:", CHUNKS_DIR)

Processed: c:\Users\MD RAJA\Quant_Research_Intelligence_System\data\processed_papers
Chunks: c:\Users\MD RAJA\Quant_Research_Intelligence_System\data\chunks


In [16]:
# Model-agnostic chunking settings.
# Notebook 03 will finalize the embedding model.
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
TARGET_CHUNK_TOKENS = 400
HARD_MAX_CHUNK_TOKENS = 512

tokenizer = AutoTokenizer.from_pretrained(
    EMBEDDING_MODEL_NAME,
    use_fast=True
)

def token_count(text):
    if not text or not text.strip():
        return 0
    return len(tokenizer.encode(text, add_special_tokens=True, truncation=False))

print("Tokenizer:", EMBEDDING_MODEL_NAME)

Tokenizer: BAAI/bge-small-en-v1.5


In [17]:
def load_document(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def clean_text(text):
    if text is None:
        return ""
    text = str(text).replace("\u00ad", "")
    return re.sub(r"\s+", " ", text).strip()

def stable_id(prefix, value):
    digest = hashlib.sha256(value.encode("utf-8")).hexdigest()[:16]
    return f"{prefix}_{digest}"

def normalize_elements(doc):
    normalized = []

    for e in doc.get("elements", []):
        normalized.append({
            "element_id": e.get("element_id"),
            "document_id": doc.get("document_id"),
            "order": e.get("order"),
            "type": e.get("type", ""),
            "page": e.get("page"),
            "bbox": e.get("bbox"),
            "section_path": e.get("section_path") or [],
            "text": clean_text(e.get("text") or ""),
            "raw": e.get("raw"),
            "caption": clean_text(e.get("caption")),
            "image_path": e.get("image_path"),
            "table_markdown": e.get("table_markdown"),
            "table_data": e.get("table_data"),
            "formula_id": e.get("formula_id"),
            "figure_id": e.get("figure_id"),
        })

    return normalized

In [18]:
SPECIAL_TYPES = {"figure", "table", "formula"}

def element_text(e):
    if e["type"] == "formula":
        content = e.get("raw") or e.get("text") or ""
    elif e["type"] == "table":
        content = e.get("table_markdown") or e.get("text") or ""
    else:
        content = e.get("text") or ""

    if e.get("caption"):
        content = e["caption"] + " " + content

    return clean_text(content)

def make_evidence(e):
    return {
        "element_id": e["element_id"],
        "type": e["type"],
        "page": e["page"],
        "bbox": e["bbox"],
        "section_path": e["section_path"],
        "caption": e["caption"],
        "image_path": e["image_path"],
        "formula_id": e["formula_id"],
        "figure_id": e["figure_id"],
    }

In [19]:
def split_sentences(text):
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", text.strip())
    return [p.strip() for p in parts if p.strip()]

def hard_split(text, max_tokens):
    words = text.split()
    pieces, current = [], []

    for word in words:
        candidate = " ".join(current + [word])

        if current and token_count(candidate) > max_tokens:
            pieces.append(" ".join(current))
            current = [word]
        else:
            current.append(word)

    if current:
        pieces.append(" ".join(current))

    return pieces

def split_long_text(text, max_tokens):
    if token_count(text) <= max_tokens:
        return [text]

    sentences = split_sentences(text)

    if len(sentences) <= 1:
        return hard_split(text, max_tokens)

    pieces, current = [], []

    for sentence in sentences:
        if token_count(sentence) > max_tokens:
            if current:
                pieces.append(" ".join(current))
                current = []
            pieces.extend(hard_split(sentence, max_tokens))
            continue

        candidate = " ".join(current + [sentence])

        if current and token_count(candidate) > max_tokens:
            pieces.append(" ".join(current))
            current = [sentence]
        else:
            current.append(sentence)

    if current:
        pieces.append(" ".join(current))

    return pieces

In [20]:
def build_text_chunks(elements):
    chunks = []
    current = None

    def flush():
        nonlocal current
        if current and current["text"].strip():
            chunks.append(current)
        current = None

    for e in elements:
        # Section boundaries are hard semantic boundaries.
        if e["type"] in {"section", "title"}:
            flush()
            continue

        # Lists and multimodal elements are preserved separately.
        if e["type"] in SPECIAL_TYPES or e["type"] == "list_item":
            flush()
            continue

        if not e["text"]:
            continue

        for piece in split_long_text(e["text"], HARD_MAX_CHUNK_TOKENS):
            if current is None:
                current = {
                    "text": piece,
                    "element_ids": [e["element_id"]],
                    "elements": [e],
                    "section_path": e["section_path"],
                }
                continue

            candidate = current["text"] + " " + piece
            same_section = current["section_path"] == e["section_path"]

            if same_section and token_count(candidate) <= TARGET_CHUNK_TOKENS:
                current["text"] = candidate
                current["element_ids"].append(e["element_id"])
                current["elements"].append(e)
            else:
                flush()
                current = {
                    "text": piece,
                    "element_ids": [e["element_id"]],
                    "elements": [e],
                    "section_path": e["section_path"],
                }

    flush()
    return chunks

In [21]:
def attach_special_evidence(elements, text_chunks):
    output = list(text_chunks)

    for e in elements:
        if e["type"] not in SPECIAL_TYPES:
            continue

        content = element_text(e)

        if not content and not e.get("image_path"):
            continue

        output.append({
            "text": content,
            "element_ids": [e["element_id"]],
            "elements": [e],
            "section_path": e["section_path"],
            "chunk_type": f"{e['type']}_evidence",
        })

    return output

In [22]:
def finalize_chunks(doc, raw_chunks):
    final = []

    for idx, c in enumerate(raw_chunks):
        elements = c["elements"]
        pages = sorted({
            e["page"] for e in elements
            if e.get("page") is not None
        })

        content_types = sorted({
            e["type"] for e in elements
        })

        evidence = [make_evidence(e) for e in elements]

        image_paths = [
            e["image_path"] for e in elements
            if e.get("image_path")
        ]

        figure_ids = [
            e["element_id"] for e in elements
            if e["type"] == "figure"
        ]

        table_ids = [
            e["element_id"] for e in elements
            if e["type"] == "table"
        ]

        formula_ids = [
            e["element_id"] for e in elements
            if e["type"] == "formula"
        ]

        chunk_type = c.get("chunk_type", "semantic")
        source_key = f"{doc.get('document_id')}|{idx}|{c['text']}"

        final.append({
            "chunk_id": stable_id("chunk", source_key),
            "document_id": doc.get("document_id"),
            "source_file": doc.get("source_file"),
            "chunk_index": idx,
            "chunk_type": chunk_type,
            "text": c["text"],
            "retrieval_text": c["text"],
            "token_count": token_count(c["text"]),
            "char_count": len(c["text"]),
            "section_path": c["section_path"],
            "pages": pages,
            "element_ids": c["element_ids"],
            "content_types": content_types,
            "figure_ids": figure_ids,
            "table_ids": table_ids,
            "formula_ids": formula_ids,
            "image_paths": image_paths,
            "evidence": evidence,
            "previous_chunk_id": None,
            "next_chunk_id": None,
        })

    for i, c in enumerate(final):
        if i:
            c["previous_chunk_id"] = final[i - 1]["chunk_id"]
        if i + 1 < len(final):
            c["next_chunk_id"] = final[i + 1]["chunk_id"]

    return final

In [23]:
def validate_chunks(chunks, elements):
    ids = [c["chunk_id"] for c in chunks]
    source_ids = {
        e["element_id"] for e in elements
        if e.get("element_id")
    }

    covered = {
        eid for c in chunks for eid in c["element_ids"]
    }

    multimodal = [
        e for e in elements
        if e["type"] in SPECIAL_TYPES
    ]

    over_limit = [
        c["chunk_id"] for c in chunks
        if c["token_count"] > HARD_MAX_CHUNK_TOKENS
    ]

    return {
        "chunk_count": len(chunks),
        "duplicate_chunk_ids": len(ids) - len(set(ids)),
        "empty_chunks": sum(not c["text"].strip() for c in chunks),
        "over_hard_token_limit": len(over_limit),
        "source_elements": len(source_ids),
        "covered_elements": len(covered & source_ids),
        "uncovered_elements": len(source_ids - covered),
        "multimodal_elements": len(multimodal),
        "multimodal_covered": sum(
            e["element_id"] in covered for e in multimodal
        ),
        "all_valid": (
            len(ids) == len(set(ids))
            and not any(not c["text"].strip() for c in chunks)
            and not over_limit
        ),
    }

In [24]:
# Test one structured document first
json_files = sorted(PROCESSED_DIR.glob("*.json"))
assert json_files, f"No structured JSON files found in {PROCESSED_DIR}"

test_file = json_files[0]
doc = load_document(test_file)
elements = normalize_elements(doc)

raw_chunks = build_text_chunks(elements)
raw_chunks = attach_special_evidence(elements, raw_chunks)

chunks = finalize_chunks(doc, raw_chunks)
qa = validate_chunks(chunks, elements)

print("Document:", test_file.name)
print("Elements:", len(elements))
print("Chunks:", len(chunks))
print("QA:", qa)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (528 > 512). Running this sequence through the model will result in indexing errors


Document: doc_127ff1e6f5f2.json
Elements: 375
Chunks: 211
QA: {'chunk_count': 211, 'duplicate_chunk_ids': 0, 'empty_chunks': 6, 'over_hard_token_limit': 0, 'source_elements': 375, 'covered_elements': 300, 'uncovered_elements': 75, 'multimodal_elements': 100, 'multimodal_covered': 100, 'all_valid': False}


In [25]:
# Inspect chunks
for c in chunks[:5]:
    print("\n---", c["chunk_id"], "---")
    print("type:", c["chunk_type"])
    print("tokens:", c["token_count"])
    print("pages:", c["pages"])
    print("section:", c["section_path"])
    print(c["text"][:700])


--- chunk_634b3e042dffad81 ---
type: semantic
tokens: 201
pages: [1]
section: ['QUANTS AND MARKET EFFICIENCY']
NIHAD ALIYEV AND XINYI DENG University of Technology Sydney Abstract. We study the impact of quantitative trading strategies (quants) on price efficiency by developing a model that incorporates two defining features of quants: superior information processing capacity and limited strategic awareness. These forces have opposing effects on trading aggressiveness-better information pro- cessing increases it, while limited strategic awareness decreases it. As a result, an increase in quant participation deteriorates price efficiency when their information quality is low, improves it when information quality is high, and produces a non-monotonic effect in between. JEL Classification: D40, D82, G14

--- chunk_e9dbc9860c200c4a ---
type: semantic
tokens: 280
pages: [2]
section: ['1. Introduction']
Advances in artificial intelligence (AI), along with the availability of big data and co

In [26]:
# Inspect multimodal evidence
multimodal = [
    c for c in chunks
    if c["chunk_type"] != "semantic"
]

for c in multimodal[:5]:
    print("\n---", c["chunk_type"], "---")
    print("elements:", c["element_ids"])
    print("pages:", c["pages"])
    print("images:", c["image_paths"])
    print("text:", c["text"][:500])


--- formula_evidence ---
elements: ['elem_9edaa0e3dea2']
pages: [8]
images: []
text: E [(v - p(ϕi, xi, xj))xi|ϕi, xi] . (1)

--- formula_evidence ---
elements: ['elem_d89c41e927e0']
pages: [8]
images: []
text: xA = E(v | ϕA) 2λ - E(xB | ϕA) 2 , (2)

--- formula_evidence ---
elements: ['elem_1d487b01af1b']
pages: [8]
images: []
text: xA = E(v|ϕA) 2λ . (3)

--- formula_evidence ---
elements: ['elem_d3fe376e9235']
pages: [8]
images: []
text: ω = xA + γ · xQ + (1 - γ) · xD + u. (4)

--- formula_evidence ---
elements: ['elem_137468b16da3']
pages: [10]
images: []
text: αA = 1 2λ - γαQ 2 - (1 -γ)αD 2 . (5)


In [27]:
# Save test result
test_out = CHUNKS_DIR / f"{doc.get('document_id', test_file.stem)}_chunks.json"

with open(test_out, "w", encoding="utf-8") as f:
    json.dump(
        {
            "document_id": doc.get("document_id"),
            "chunks": chunks,
            "validation": qa,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Saved:", test_out)

Saved: c:\Users\MD RAJA\Quant_Research_Intelligence_System\data\chunks\doc_127ff1e6f5f2_chunks.json


In [28]:
# Full corpus processing
manifest = []
all_chunks = []

for path in json_files:
    doc = load_document(path)
    elements = normalize_elements(doc)

    raw_chunks = build_text_chunks(elements)
    raw_chunks = attach_special_evidence(elements, raw_chunks)

    chunks = finalize_chunks(doc, raw_chunks)
    qa = validate_chunks(chunks, elements)

    out_path = CHUNKS_DIR / f"{doc.get('document_id', path.stem)}_chunks.json"

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "document_id": doc.get("document_id"),
                "chunks": chunks,
                "validation": qa,
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

    all_chunks.extend(chunks)

    manifest.append({
        "document_id": doc.get("document_id"),
        "source_file": doc.get("source_file"),
        "chunk_count": len(chunks),
        "validation": qa,
        "output": str(out_path),
    })

manifest_path = CHUNKS_DIR / "chunk_manifest.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("Documents:", len(manifest))
print("Total chunks:", len(all_chunks))
print("Manifest:", manifest_path)

Documents: 11
Total chunks: 1141
Manifest: c:\Users\MD RAJA\Quant_Research_Intelligence_System\data\chunks\chunk_manifest.json


## Notebook 03 contract

- `retrieval_text` → embeddings + BM25
- `token_count` → chunk QA
- `evidence` → page/bbox/figure/table/formula traceability
- `image_paths` → multimodal retrieval/display
- `section_path` → structure-aware retrieval
- `element_ids` → source traceability
- KG can use `document → section → chunk → element/entity` relationships

**Important:** tokenization is always required before embedding. The token-limit check only determines whether a chunk must be split further.